# Lesson 10 · Build your own toolkit (a Python ebbr)

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 11 (pp. 100–114). The book introduces Robinson's R package `ebbr`; here you write the Python equivalent.

**Question:** Can everything so far fit in a small, reusable Python module, so the book's main results take a few lines each?

**Goal:** Write `my_eb.py`, with functions to fit a prior, add estimates and intervals, run proportion tests and fit mixtures, then use it to reproduce Chapter 11 as a review of Lessons 02–08.

**Contents**

1. The design: what the module should do
2. fit_prior and augment
3. Hierarchical priors (regression)
4. Hypothesis tests: PEP, q-values, A/B
5. Mixtures
6. Reproduce Chapter 11 in a few lines

**Tags:** `software-design` `python-module` `api-design` `testing` `review` `empirical-bayes`

**Before you start:** Lessons 02–08. Lessons 11 and 12 import this module.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

# Reload my_eb automatically every time you edit and save it
%load_ext autoreload
%autoreload 2

career_full = eb_data.career(pitcher_rule=3, extra=True).dropna(subset=["bats"])
career = career_full.drop(columns=["bats", "year"])
print(f"{len(career):,} players")

## 1. The design: what the module should do

Create **`my_eb.py`** in this folder (File → New → Python File in JupyterLab) and develop it function by function, testing each one here. A suggested interface, loosely following `ebbr`:

| Function | Does | Returns |
|---|---|---|
| `fit_prior(df, x, n, subset=None)` | MLE beta prior from columns `x` (successes) and `n` (totals), optionally on a subset of rows | a small object or dict with `alpha`, `beta`, `mean` |
| `augment(prior, df, x, n, cred_level=0.95)` | add per-row `alpha0, beta0, alpha1, beta1, fitted, raw, low, high` | a new DataFrame |
| `add_estimate(df, x, n, subset=None, **kw)` | `fit_prior` then `augment`, in one call | a new DataFrame |
| `fit_prior_regression(df, x, n, X)` | beta-binomial regression (Lesson 06), with a design matrix | coefficients, σ, and a way to predict μ for new data |
| `add_prop_test(df, threshold, approx=False)` | `pep` and `qvalue` against a fixed threshold *or* another posterior `(alpha, beta)` | a new DataFrame |
| `fit_mixture(df, x, n, k=2, iterations=10)` | EM for a beta-binomial mixture (Lesson 08) | parameters, assignments, per-iteration history |

Pandas tip: `ebbr` puts a leading dot in its new column names (`.fitted`). Plain names like `fitted` are easier to work with in pandas.

Write a docstring for each function. It's good practice and it'll help when you come back later.

## 2. fit_prior and augment

**Your turn.** Write `fit_prior` and `augment` in `my_eb.py`, then import them here and fit on players with AB ≥ 500.

In [ ]:
# Your code here

> **Check:** with this chapter's data (pitchers with more than 3 games removed, missing handedness dropped) the book got α ≈ 96.9, β ≈ 275.

**Your turn.** Using only the augmented DataFrame and matplotlib, recreate two figures: raw vs shrunken (coloured by AB), and the 1998 Yankees' intervals.

In [ ]:
# Your code here

In [ ]:
# Your code here

## 3. Hierarchical priors (regression)

**Your turn.** Add `fit_prior_regression`, and let `augment` accept it so that each row gets its own `alpha0`, `beta0`. Fit μ on `log10(AB)`, then plot raw vs shrunken estimates against AB, side by side.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Your turn.** Fit the full prior from Lesson 07, with spline(year) × bats + log(AB). Using made-up players (H = 300, AB = 1,000, every year 1885–2013, L and R), plot the prior mean and 95% range by year.

In [ ]:
# Your code here

## 4. Hypothesis tests: PEP, q-values, A/B

**Your turn.** Write `add_prop_test`. Test against .300 using the log10(AB) prior. How many players are admitted at q < 0.05 and at q < 0.01?

In [ ]:
# Your code here

> **Check:** the book got 115 at 5% and 79 at 1% (with the AB-dependent prior, so more than Lesson 04).

**Your turn.** Let `add_prop_test` take `(alpha, beta)` instead of a threshold, so it compares every player with one posterior (use the normal approximation when `approx=True`). Compare everyone with Mike Piazza.

In [ ]:
# Your code here

> **Check:** Piazza's posterior under the log10(AB) prior ≈ Beta(2281, 5183).

## 5. Mixtures

**Your turn.** Write `fit_mixture` from your Lesson 08 code. Fit it to `eb_data.career_with_pitchers(min_ab=24)`, i.e. AB ≥ 25, as in the book, and plot the assignments and the parameter traces across iterations.

In [ ]:
# Your code here

In [ ]:
# Your code here

> **Check:** clusters near Beta(25.8, 152) (mean 0.145) and Beta(161, 460) (mean 0.259).

## 6. Reproduce Chapter 11 in a few lines

**Your turn.** In a single cell, starting from `career`, go from raw data to a sorted Hall of Fame table (name, H, AB, fitted, low, high, pep, qvalue) using only `my_eb` functions.

In [ ]:
# Your code here

**Reflect.** What was hardest to design: the inputs, the outputs, or the edge cases (AB = 0, H = AB, a failed fit)? What would you add before using `my_eb.py` on the card data?

*Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 11.
- ebbr source (R): https://github.com/dgrtwo/ebbr
- Python Packaging User Guide: https://packaging.python.org/